# MLB Salary Cap Analysis - Case Study 3

## Research Question
With MLB owners proposing a \\$245.3M cap and \\$171.2M floor for the 2027 season, 
which teams face the greatest structural pressure on either end — and what does 
20 years of World Series history suggest about what's actually at stake competitively?

## Structure
1. Cap Pressure - teams currently above \$245.3M
2. Floor Pressure - teams currently under \$171.2M
3. World Series History - payroll of champions over last 20 years
4. Closing Thesis - asymmetric pressure analysis

In [6]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd        # tabular data — loading, filtering, merging, transforming
import numpy as np         # numerical operations and array math
import matplotlib.pyplot as plt  # base visualization library
import seaborn as sns      # statistical visualizations, cleaner defaults than raw matplotlib
import pybaseball as pb    # pulls data from Baseball Reference, FanGraphs, Statcast
import requests

## Section 1 — Cap Pressure

Identifying which MLB teams currently exceed the proposed \$245.3M salary cap, 
and quantifying how much they would need to shed to comply.

### Data Source — 2026 Team Payrolls

Current team payroll data is sourced from Spotrac, which tracks real-time 
MLB salary commitments. Since no direct download is available, we scrape 
the data programmatically using pandas' `read_html()` function. We're using 
2026 figures as the most relevant proxy for evaluating pressure against the 
proposed 2027 salary cap and floor thresholds.

In [7]:
# url = 'https://www.spotrac.com/mlb/payroll/_/year/2026'
# tables = pd.read_html(url)
# print(len(tables))

# headers = {'User-Agent': 'Mozilla/5.0'}
# response = requests.get(url, headers=headers)
# tables = pd.read_html(response.text)
# print(len(tables))

FileNotFoundError: [Errno 2] No such file or directory: <!DOCTYPE html>
<html lang="en">
  
	<head>
	<meta charset="utf-8">
	<title>2026 MLB Team Salary Payroll Tracker</title>
	
    <!-- SEO Meta Tags-->
	<meta name="description" content="A real-time look at the 2026 salary payroll totals for each MLB team..">
	<meta name="keywords" content="2026, MLB, payroll, space, team, rankings">
	<meta name="author" content="Spotrac">
	
    <!-- Viewport-->
	<meta name="viewport" content="width=device-width, initial-scale=1">
    
    <meta property="fb:app_id" content="206107179423868" />
    <meta property="fb:admins" content="513540967"/>
    <meta property="og:site_name" content="spotrac.com" />
    <meta property="og:url" content="https://www.spotrac.com/mlb/payroll/_/year/2026" />
    <meta property="og:title" content="2026 MLB Team Salary Payroll Tracker"/>
        <meta property="og:description" content="A real-time look at the 2026 salary payroll totals for each MLB team.." />
        <

### A Note on Data Collection

Initial efforts to scrape 2026 team payroll figures directly from Spotrac 
were unsuccessful due to the site's use of JavaScript to render its data 
tables. Attempts using `requests` and `pd.read_html()` retrieved only 
static HTML — the payroll table itself is loaded dynamically and requires 
a browser to execute. A full scraping solution would utilize Python's 
`Selenium` package to control a headless browser and capture the rendered 
output. For simplicity, 2026 payroll figures were manually collected from 
Spotrac and loaded via CSV.

In [9]:
payroll_df = pd.read_csv('spotrac-team-payroll-2026.csv')
payroll_df.head()

,Team,Total Payroll
0,PHI,"280,839,235"
1,TOR,"282,552,776"
2,NYY,"292,892,832"
3,NYM,"334,085,472"
4,ATL,"267,889,815"


In [10]:
payroll_df.dtypes

Team             str
Total Payroll    str
dtype: object